In [ ]:
!pip install pyspark

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("OnlineRetailAnalysis") \
    .getOrCreate()

In [ ]:
df = spark.read.csv("/OnlineRetail.csv", header=True, inferSchema=True)
df.show(5)

+---------+---------+--------------------+--------+--------------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity|   InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+--------------+---------+----------+--------------+
|   536365|   85123A|WHITE HANGING HEA...|       6|12/1/2010 8:26|     2.55|     17850|United Kingdom|
|   536365|    71053| WHITE METAL LANTERN|       6|12/1/2010 8:26|     3.39|     17850|United Kingdom|
|   536365|   84406B|CREAM CUPID HEART...|       8|12/1/2010 8:26|     2.75|     17850|United Kingdom|
|   536365|   84029G|KNITTED UNION FLA...|       6|12/1/2010 8:26|     3.39|     17850|United Kingdom|
|   536365|   84029E|RED WOOLLY HOTTIE...|       6|12/1/2010 8:26|     3.39|     17850|United Kingdom|
+---------+---------+--------------------+--------+--------------+---------+----------+--------------+
only showing top 5 rows


In [ ]:
df = df.dropna()
df = df.dropDuplicates()

df = df.filter(df["Quantity"] > 0)
df = df.filter(df["UnitPrice"] > 0)

df.show(5)

+---------+---------+--------------------+--------+---------------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity|    InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+---------------+---------+----------+--------------+
|   536381|    21731|RED TOADSTOOL LED...|       2| 12/1/2010 9:41|     1.65|     15311|United Kingdom|
|   536390|    22174|          PHOTO CUBE|      48|12/1/2010 10:19|     1.48|     17511|United Kingdom|
|   536396|    82483|WOOD 2 DRAWER CAB...|       2|12/1/2010 10:51|     4.95|     17850|United Kingdom|
|   536408|    22914|BLUE COAT RACK PA...|       3|12/1/2010 11:41|     4.95|     14307|United Kingdom|
|   536412|    22382|LUNCH BAG SPACEBO...|       3|12/1/2010 11:49|     1.65|     17920|United Kingdom|
+---------+---------+--------------------+--------+---------------+---------+----------+--------------+
only showing top 5 rows


In [ ]:
from pyspark.sql.functions import col

df = df.withColumn("TotalPrice", col("Quantity") * col("UnitPrice"))
df.show(5)

+---------+---------+--------------------+--------+---------------+---------+----------+--------------+------------------+
|InvoiceNo|StockCode|         Description|Quantity|    InvoiceDate|UnitPrice|CustomerID|       Country|        TotalPrice|
+---------+---------+--------------------+--------+---------------+---------+----------+--------------+------------------+
|   536381|    21731|RED TOADSTOOL LED...|       2| 12/1/2010 9:41|     1.65|     15311|United Kingdom|               3.3|
|   536390|    22174|          PHOTO CUBE|      48|12/1/2010 10:19|     1.48|     17511|United Kingdom| 71.03999999999999|
|   536396|    82483|WOOD 2 DRAWER CAB...|       2|12/1/2010 10:51|     4.95|     17850|United Kingdom|               9.9|
|   536408|    22914|BLUE COAT RACK PA...|       3|12/1/2010 11:41|     4.95|     14307|United Kingdom|14.850000000000001|
|   536412|    22382|LUNCH BAG SPACEBO...|       3|12/1/2010 11:49|     1.65|     17920|United Kingdom| 4.949999999999999|
+---------+-----

In [ ]:
df.filter(df["Country"] == "United Kingdom").show(5)

df.groupBy("Country").sum("TotalPrice").show()

+---------+---------+--------------------+--------+---------------+---------+----------+--------------+------------------+
|InvoiceNo|StockCode|         Description|Quantity|    InvoiceDate|UnitPrice|CustomerID|       Country|        TotalPrice|
+---------+---------+--------------------+--------+---------------+---------+----------+--------------+------------------+
|   536381|    21731|RED TOADSTOOL LED...|       2| 12/1/2010 9:41|     1.65|     15311|United Kingdom|               3.3|
|   536390|    22174|          PHOTO CUBE|      48|12/1/2010 10:19|     1.48|     17511|United Kingdom| 71.03999999999999|
|   536396|    82483|WOOD 2 DRAWER CAB...|       2|12/1/2010 10:51|     4.95|     17850|United Kingdom|               9.9|
|   536408|    22914|BLUE COAT RACK PA...|       3|12/1/2010 11:41|     4.95|     14307|United Kingdom|14.850000000000001|
|   536412|    22382|LUNCH BAG SPACEBO...|       3|12/1/2010 11:49|     1.65|     17920|United Kingdom| 4.949999999999999|
+---------+-----

In [ ]:
df.count()
df.show(10)

+---------+---------+--------------------+--------+---------------+---------+----------+--------------+------------------+
|InvoiceNo|StockCode|         Description|Quantity|    InvoiceDate|UnitPrice|CustomerID|       Country|        TotalPrice|
+---------+---------+--------------------+--------+---------------+---------+----------+--------------+------------------+
|   536381|    21731|RED TOADSTOOL LED...|       2| 12/1/2010 9:41|     1.65|     15311|United Kingdom|               3.3|
|   536390|    22174|          PHOTO CUBE|      48|12/1/2010 10:19|     1.48|     17511|United Kingdom| 71.03999999999999|
|   536396|    82483|WOOD 2 DRAWER CAB...|       2|12/1/2010 10:51|     4.95|     17850|United Kingdom|               9.9|
|   536408|    22914|BLUE COAT RACK PA...|       3|12/1/2010 11:41|     4.95|     14307|United Kingdom|14.850000000000001|
|   536412|    22382|LUNCH BAG SPACEBO...|       3|12/1/2010 11:49|     1.65|     17920|United Kingdom| 4.949999999999999|
|   536416|    2

In [ ]:
from pyspark.sql.functions import avg, max

df.groupBy("Country").agg(
    avg("TotalPrice"),
    max("TotalPrice")
).show()

+------------------+------------------+------------------+
|           Country|   avg(TotalPrice)|   max(TotalPrice)|
+------------------+------------------+------------------+
|            Sweden| 94.75035460992908|            1188.0|
|         Singapore|  90.8199115044248|           2382.92|
|           Germany|26.296530993278605| 525.5999999999999|
|            France| 24.75321535181242|            1136.3|
|            Greece| 43.03870588235294|             175.2|
|European Community|17.400000000000002|              36.0|
|           Belgium|19.255283363802544|104.64999999999999|
|           Finland| 32.69632653061224|             551.2|
|       Unspecified|          18.69375|              30.0|
|             Italy| 22.61146616541354|             196.0|
|              EIRE| 40.27023622047249|            1752.0|
|         Lithuania| 47.45885714285714|122.39999999999999|
|            Norway| 28.13529702970298|             376.5|
|             Spain| 25.33471380471381|            1220.

In [ ]:
df.createOrReplaceTempView("retail")
spark.sql("""
SELECT Country, SUM(TotalPrice) as TotalSales
FROM retail
GROUP BY Country
ORDER BY TotalSales DESC
""").show()

+---------------+------------------+
|        Country|        TotalSales|
+---------------+------------------+
| United Kingdom|2232045.6509998576|
|    Netherlands| 83800.59999999998|
|           EIRE| 71600.48000000008|
|        Germany|  70422.1100000001|
|         France| 58046.29000000012|
|      Australia| 42674.07000000002|
|          Spain|22573.230000000007|
|          Japan|20083.280000000002|
|         Sweden|           13359.8|
|    Switzerland|12138.240000000002|
|       Portugal|12058.179999999997|
|        Belgium|10532.639999999992|
|      Singapore|10262.650000000001|
|        Finland| 9612.719999999998|
|         Cyprus| 7407.199999999997|
|Channel Islands| 6626.150000000001|
|          Italy|6014.6500000000015|
|         Norway| 5683.330000000002|
|        Denmark|           5659.71|
|         Greece|           3658.29|
+---------------+------------------+
only showing top 20 rows


In [ ]:
from pyspark.sql.functions import when

df = df.withColumn("label", when(df["Quantity"] > 10, 1).otherwise(0))

from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=["UnitPrice", "Quantity"],
    outputCol="features"
)

data = assembler.transform(df)

from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(featuresCol="features", labelCol="label")
model = lr.fit(data)

predictions = model.transform(data)
predictions.select("features", "label", "prediction").show()

+-----------+-----+----------+
|   features|label|prediction|
+-----------+-----+----------+
| [1.65,2.0]|    0|       0.0|
|[1.48,48.0]|    1|       1.0|
| [4.95,2.0]|    0|       0.0|
| [4.95,3.0]|    0|       0.0|
| [1.65,3.0]|    0|       0.0|
| [9.95,4.0]|    0|       0.0|
| [2.95,6.0]|    0|       0.0|
| [1.65,8.0]|    0|       0.0|
| [4.95,2.0]|    0|       0.0|
|[1.65,12.0]|    1|       1.0|
| [3.95,1.0]|    0|       0.0|
|[0.65,12.0]|    1|       1.0|
| [7.95,4.0]|    0|       0.0|
|  [2.1,6.0]|    0|       0.0|
|[1.25,12.0]|    1|       1.0|
|[1.25,12.0]|    1|       1.0|
| [3.75,4.0]|    0|       0.0|
| [4.95,6.0]|    0|       0.0|
|[1.65,12.0]|    1|       1.0|
| [0.65,1.0]|    0|       0.0|
+-----------+-----+----------+
only showing top 20 rows
